# GitHub仓库关联度分析

根据关联度.docx文档的要求，实现多维度关联度计算：
1. 生态关联度（基于贡献者、语言、主题等）
2. 语义关联度（基于文本向量化）
3. 时序关联度（基于时间戳）

最终融合：Rtotal = 0.5 × Reco + 0.3 × Rsemantic + 0.2 × Rtemporal

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 1. 加载数据

In [ ]:
with open('3.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

repos_data = data['repos_data']
print(f"Total repositories: {len(repos_data)}")

## 2. 数据预处理和特征提取

In [ ]:
def extract_features(repo):
    """提取仓库特征用于关联度计算"""
    basic = repo['basic_info']
    metrics = repo['metrics']
    activity = repo['activity']
    timeline = repo['timeline']
    
    features = {
        'repo_name': basic['full_name'],
        'repo_id': basic['repo_id'],
        'owner': basic['owner'],
        'description': basic.get('description', ''),
        'language': basic.get('language', ''),
        'topics': basic.get('topics', []),
        'license': basic.get('license', ''),
        'star_count': metrics['star_count'],
        'fork_count': metrics['fork_count'],
        'subscriber_count': metrics['subscriber_count'],
        'open_issues_count': metrics['open_issues_count'],
        'watchers_count': metrics['watchers_count'],
        'size_kb': metrics['size_kb'],
        'commits_total': activity['commits_total'],
        'prs_total': activity['prs_total'],
        'contributors_total': activity['contributors_total'],
        'activity_score': activity['activity_score'],
        'created_at': timeline['created_at'],
        'updated_at': timeline['updated_at'],
        'pushed_at': timeline['pushed_at'],
        'readme_content': repo.get('readme_content', '')
    }
    return features

features_list = [extract_features(repo) for repo in repos_data]
df = pd.DataFrame(features_list)
print(f"Features extracted: {df.shape[1]} columns")
print(f"\nFirst few columns: {list(df.columns[:10])}")

## 3. 保存为CSV文件

In [ ]:
df.to_csv('github_repos_features.csv', index=False, encoding='utf-8-sig')
print("CSV file saved: github_repos_features.csv")
print(f"Shape: {df.shape}")
df.head(3)

## 4. 文本预处理

In [ ]:
def clean_text(text):
    """清理文本数据"""
    if not text:
        return ''
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)  # 移除HTML标签
    text = re.sub(r'\[.*?\]\(.*?\)', '', text)  # 移除Markdown链接
    text = re.sub(r'[^\w\s]', ' ', text)  # 移除特殊字符
    text = re.sub(r'\s+', ' ', text)  # 合并空格
    return text.strip().lower()

def combine_text_features(row):
    """组合文本特征"""
    parts = []
    
    if row['description']:
        parts.append(clean_text(row['description']))
    
    if row['topics'] and isinstance(row['topics'], list):
        parts.append(' '.join(row['topics']))
    
    if row['language']:
        parts.append(row['language'].lower())
    
    if row['readme_content']:
        readme_clean = clean_text(row['readme_content'])
        parts.append(readme_clean[:2000])  # 限制README长度
    
    return ' '.join(parts)

df['combined_text'] = df.apply(combine_text_features, axis=1)
print("Text features combined")
print(f"Sample combined text: {df['combined_text'].iloc[0][:200]}...")

## 5. 语义关联度计算（基于TF-IDF向量化）

In [ ]:
def calculate_semantic_similarity(texts):
    """计算语义相似度矩阵"""
    vectorizer = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.8,
        stop_words='english'
    )
    
    tfidf_matrix = vectorizer.fit_transform(texts)
    similarity_matrix = cosine_similarity(tfidf_matrix)
    
    return similarity_matrix

semantic_matrix = calculate_semantic_similarity(df['combined_text'].tolist())
print(f"Semantic similarity matrix shape: {semantic_matrix.shape}")
print(f"Sample similarity values: {semantic_matrix[0, :5]}")

## 6. 生态关联度计算（基于数值特征）

In [ ]:
def calculate_ecological_similarity(df):
    """计算生态关联度"""
    numeric_features = [
        'star_count',
        'fork_count',
        'subscriber_count',
        'contributors_total',
        'activity_score'
    ]
    
    X = df[numeric_features].values
    
    scaler = MinMaxScaler()
    X_normalized = scaler.fit_transform(X)
    
    similarity_matrix = cosine_similarity(X_normalized)
    
    return similarity_matrix

eco_matrix = calculate_ecological_similarity(df)
print(f"Ecological similarity matrix shape: {eco_matrix.shape}")
print(f"Sample similarity values: {eco_matrix[0, :5]}")

## 7. 时序关联度计算（基于创建时间）

In [ ]:
def calculate_temporal_similarity(df):
    """计算时序关联度（基于创建时间）"""
    created_dates = pd.to_datetime(df['created_at'])
    
    n = len(df)
    temporal_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i == j:
                temporal_matrix[i, j] = 1.0
            else:
                days_diff = abs((created_dates[i] - created_dates[j]).days)
                temporal_matrix[i, j] = np.exp(-days_diff / 365)  # 1年衰减
    
    return temporal_matrix

temporal_matrix = calculate_temporal_similarity(df)
print(f"Temporal similarity matrix shape: {temporal_matrix.shape}")
print(f"Sample similarity values: {temporal_matrix[0, :5]}")

## 8. 融合多维度关联度

In [ ]:
def calculate_total_similarity(eco_matrix, semantic_matrix, temporal_matrix, 
                               w_eco=0.5, w_semantic=0.3, w_temporal=0.2):
    """融合多维度关联度"""
    total_matrix = (w_eco * eco_matrix + 
                   w_semantic * semantic_matrix + 
                   w_temporal * temporal_matrix)
    
    return total_matrix

total_matrix = calculate_total_similarity(eco_matrix, semantic_matrix, temporal_matrix)
print(f"Total similarity matrix shape: {total_matrix.shape}")
print(f"Sample similarity values: {total_matrix[0, :5]}")
print(f"Min similarity: {total_matrix.min():.4f}")
print(f"Max similarity: {total_matrix.max():.4f}")
print(f"Mean similarity: {total_matrix.mean():.4f}")

## 9. 保存关联度矩阵

In [ ]:
np.save('repo_similarity_matrix.npy', total_matrix)
print("Similarity matrix saved: repo_similarity_matrix.npy")

similarity_df = pd.DataFrame(
    total_matrix,
    index=df['repo_name'].values,
    columns=df['repo_name'].values
)
similarity_df.to_csv('repo_similarity_matrix.csv', index=True, encoding='utf-8-sig')
print("Similarity matrix CSV saved: repo_similarity_matrix.csv")

## 10. 查找最相似的仓库对

In [ ]:
def find_top_similar_pairs(similarity_matrix, repo_names, top_n=20):
    """查找最相似的仓库对"""
    pairs = []
    n = len(repo_names)
    
    for i in range(n):
        for j in range(i+1, n):
            pairs.append({
                'repo1': repo_names[i],
                'repo2': repo_names[j],
                'similarity': similarity_matrix[i, j]
            })
    
    pairs_df = pd.DataFrame(pairs)
    pairs_df = pairs_df.sort_values('similarity', ascending=False).head(top_n)
    
    return pairs_df

top_pairs = find_top_similar_pairs(total_matrix, df['repo_name'].values, top_n=20)
print("Top 20 most similar repository pairs:")
top_pairs

## 11. 为特定仓库查找相似仓库

In [ ]:
def find_similar_repos(repo_name, similarity_df, top_n=10):
    """为特定仓库查找最相似的仓库"""
    if repo_name not in similarity_df.index:
        print(f"Repository '{repo_name}' not found!")
        return None
    
    similarities = similarity_df[repo_name].sort_values(ascending=False)
    similarities = similarities[similarities.index != repo_name].head(top_n)
    
    return similarities

example_repo = df['repo_name'].iloc[0]
similar_repos = find_similar_repos(example_repo, similarity_df, top_n=10)
print(f"Top 10 repositories similar to '{example_repo}':")
similar_repos

## 12. 可视化关联度矩阵（热力图）

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 10))
sns.heatmap(
    total_matrix[:50, :50],
    cmap='YlOrRd',
    xticklabels=False,
    yticklabels=False,
    cbar_kws={'label': 'Similarity Score'}
)
plt.title('Repository Similarity Matrix (First 50 Repositories)')
plt.xlabel('Repository Index')
plt.ylabel('Repository Index')
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Heatmap saved: similarity_heatmap.png")

## 13. 统计分析

In [ ]:
print("=== Similarity Statistics ===")
print(f"Mean similarity: {total_matrix.mean():.4f}")
print(f"Std similarity: {total_matrix.std():.4f}")
print(f"Median similarity: {np.median(total_matrix):.4f}")
print(f"Min similarity: {total_matrix.min():.4f}")
print(f"Max similarity: {total_matrix.max():.4f}")

print("\n=== Similarity Distribution ===")
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
hist, _ = np.histogram(total_matrix, bins=bins)
for i in range(len(bins)-1):
    print(f"[{bins[i]:.1f}, {bins[i+1]:.1f}): {hist[i]} pairs")

## 14. 生成最终结果文件

In [ ]:
result = {
    "metadata": {
        "total_repos": len(df),
        "calculation_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "weights": {
            "ecological": 0.5,
            "semantic": 0.3,
            "temporal": 0.2
        },
        "statistics": {
            "mean_similarity": float(total_matrix.mean()),
            "std_similarity": float(total_matrix.std()),
            "min_similarity": float(total_matrix.min()),
            "max_similarity": float(total_matrix.max())
        }
    },
    "repositories": df['repo_name'].tolist(),
    "similarity_matrix": total_matrix.tolist(),
    "top_similar_pairs": top_pairs.to_dict('records')
}

with open('repo_similarity_result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("Final result saved: repo_similarity_result.json")
print(f"\nSummary:")
print(f"- Total repositories: {result['metadata']['total_repos']}")
print(f"- Mean similarity: {result['metadata']['statistics']['mean_similarity']:.4f}")
print(f"- Top similar pairs: {len(result['top_similar_pairs'])}")

## 15. 完成总结

In [ ]:
print("=== Analysis Complete ===")
print("\nGenerated files:")
print("1. github_repos_features.csv - Repository features in CSV format")
print("2. repo_similarity_matrix.npy - Similarity matrix (NumPy format)")
print("3. repo_similarity_matrix.csv - Similarity matrix (CSV format)")
print("4. repo_similarity_result.json - Complete results with metadata")
print("5. similarity_heatmap.png - Visualization of similarity matrix")
print("\nKey findings:")
print(f"- Analyzed {len(df)} repositories")
print(f"- Calculated {len(df) * (len(df) - 1) // 2} pairwise similarities")
print(f"- Average similarity: {total_matrix.mean():.4f}")
print(f"- Top similarity: {total_matrix.max():.4f}")